# Transformer

- Embedding + Positional Encoding  
    - ↓  
- Encoder (N개 층) → EncoderLayer = MultiHeadAttention + LayerNorm + FFN + LayerNorm  
    - ↓  
- Decoder (N개 층) → DecoderLayer = MaskedAttention + LayerNorm + CrossAttention + LayerNorm + FFN + LayerNorm  
    - ↓  
- Linear + Softmax (출력)  

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

### Embedding

- 1단계 토큰화 : 문장을 쪼개서 각 조각에 고유번호를 매기는 것 (BPE로 실습)
- 2단계 임베딩 : 토큰화에서 붙인 고유번호를 의미를 담을 수 있는 벡터로 바꿔주는 것 

In [ ]:
vocab_size = 10000
d_model = 512

# 룩업 테이블 생성
embedding_layer = nn.Embedding(vocab_size, d_model)

input_ids = torch.tensor([[15, 302, 891, 5]])
# id를 벡터로 변환
output = embedding_layer(input_ids)

print(output.shape)

torch.Size([1, 4, 512])


In [5]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

In [9]:
# 테스트
# 1. 클래스로 인스턴스 만들기
token_embedding = TokenEmbedding(vocab_size, d_model)

# 2. 실제로 돌려보기
output2 = token_embedding(input_ids)

print(output2.shape)

print(output[0][0][:5])    # 스케일링 안 한 버전 (앞 5개 값)
print(output2[0][0][:5])   # 스케일링 한 버전 (앞 5개 값)

torch.Size([1, 4, 512])
tensor([-0.2082,  0.5177,  0.7930,  0.7129, -0.1090], grad_fn=<SliceBackward0>)
tensor([ 46.6920,   5.9588,  -8.6954, -23.3507,  -3.6008],
       grad_fn=<SliceBackward0>)


## Token Embedding 구현 정리

### 목적
토큰 ID(정수) → 의미를 담은 벡터로 변환

### 핵심 코드
```python
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)
```

### 파라미터
- `vocab_size`: 사전 크기 (BPE 결과에서 세어서 나옴, 데이터 종속)
- `d_model`: 벡터 차원 (설계 시 자유 선택, 논문 기준 512)

### 스케일링 이유
- 임베딩 결과에 $\sqrt{d_{model}}$을 곱함
- 이유: 나중에 Positional Encoding(-1~1 범위)을 더할 때 임베딩 값이 묻히지 않게 크기 맞춤

### shape 흐름
$$(\text{batch}, \text{seq\_len}) \xrightarrow{\text{Embedding}} (\text{batch}, \text{seq\_len}, d_{model})$$

### 확인한 것
- `output.shape` = `torch.Size([1, 4, 512])` ✅
- 클래스 버전(`output2`)이 원본(`output`)보다 절댓값이 훨씬 큼 -> 스케일링 정상 작동 확인


### Posisional Encoding
- Attention은 누가 몇 번째 자리에 있는지 순서 개념이 없기 때문에 PE가 각 토큰에게 번호표(벡터)를 만들어서 임베딩에 더해준다. 
- pos = 몇 번째 토큰인지 (0,1,2,3...)
- i = 벡터의 몇 번째 차원인지
- 짝수 차원엔 sin, 홀수 차원엔 cos 사용

- $PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$
- $PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$

In [12]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 인덱스
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 인덱스 

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

In [14]:
# 테스트
# 1. 인스턴스 만들기
pos_encoding = PositionalEncoding(d_model)

# 2. 실제로 돌려보기 
output3 = pos_encoding(output2)

print(output3.shape)

print(output2[0][0][:5])   # PE 더하기 전
print(output3[0][0][:5])   # PE 더한 후

torch.Size([1, 4, 512])
tensor([ 46.6920,   5.9588,  -8.6954, -23.3507,  -3.6008],
       grad_fn=<SliceBackward0>)
tensor([ 46.6920,   6.9588,  -8.6954, -22.3507,  -3.6008],
       grad_fn=<SliceBackward0>)


### Multihead Attention

- Self-Attention : 문장 전체를 $d_model$ 벡터 그대로 놓고 딱 한번만 이 계산을함
    - 이 문장에서 단어들끼리 어떤 관계인지 1가지 관점으로만 봄
    - $\text{Attention}(Q,K,V) = \text{softmax}(QK^T/\sqrt{d_k})V$

- Multi_Head Attention : $d_model$을 $n_heads$개로 쪼개서, 각자 $d_k$ 차원끼리 작은 어텐션을 독립적으로 계산하고, 마지막에 다시 합침

In [18]:
# Self-Attention

class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)

    def forward(self, x):
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)
        attention = F.softmax(scores, dim=-1)
        output = torch.matmul(attention, V)
        return output

In [19]:
# 테스트
self_attn = SelfAttention(d_model)
output4 = self_attn(output3)   # PE까지 통과한 결과를 넣어봄
print(output4.shape)

torch.Size([1, 4, 512])


In [8]:
class LayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        # 학습 초기에 순수한 Normalization으로 동작하기 위함 
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = 1e-5

    def forward(self, x:torch.Tensor):

        # Args - x : (batch_size, seq_len, d_model)

        mean = x.mean(dim = -1, keepdim=True)
        var = x.var(dim = -1, unbiased=False, keepdim=True)
        out = (x - mean) / torch.sqrt(var + self.eps)
        out = out * self.gamma + self.beta

        return out